# 03 · Um agente real com memória

Um assistente de conversa (Ollama, `qwen2.5:14b`) que, a cada mensagem:

1. **lembra**: busca as memórias do usuário e os trechos de documentos ligados à mensagem;
2. **responde** com esse contexto no prompt de sistema, citando documentos;
3. **aprende**: entrega a troca (pergunta + resposta) ao `memories.add`, que decide o que guardar.

Depois o programa é "fechado" (o objeto é descartado) e aberto de novo numa **segunda sessão**:
o assistente continua sabendo quem é o usuário, porque a memória está na pasta.

Este é o padrão "memória como contexto": simples, previsível, uma busca por mensagem. O
notebook 04 mostra o padrão "memória como ferramenta", em que o próprio modelo decide.

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local] @ git+https://github.com/Pedro-Bianco/pi-memory"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
import httpx
from pi_memory import PiMemory
from pi_memory.llms import OllamaLLM

OLLAMA, MODELO = "http://localhost:11434", "qwen2.5:14b"
PASTA = pasta_limpa("dados_03")
FIXTURES = Path("../../eval/fixtures")

## O agente

`conversar()` é o agente inteiro. O histórico da sessão fica em memória do programa; o que deve
durar entre sessões vai para o pi-memory.

In [3]:
SISTEMA = """Você é um assistente pessoal, em português do Brasil. Responda curto.
Use as memórias e os trechos abaixo só quando ajudarem. Ao usar um trecho, cite a fonte entre
colchetes, como [contrato.pdf, página 3]. Se não souber, diga que não sabe.

Memórias sobre o usuário:
{memorias}

Trechos de documentos:
{trechos}"""


def contexto(pm, dono, mensagem):
    memorias = pm.memories.search(mensagem, owner=dono, limit=3)["results"]
    trechos = pm.documents.search(mensagem, owner=dono, limit=3)["results"]
    return SISTEMA.format(
        memorias="\n".join(f"- {m['text']}" for m in memorias) or "(nenhuma)",
        trechos="\n".join(f"[{t['citation']['label']}] {t['text']}" for t in trechos) or "(nenhum)")


def conversar(pm, dono, historico, mensagem, mostrar_aprendizado=True):
    historico.append({"role": "user", "content": mensagem})
    mensagens = [{"role": "system", "content": contexto(pm, dono, mensagem)}, *historico[-10:]]
    resposta = httpx.post(f"{OLLAMA}/api/chat", timeout=180, json={
        "model": MODELO, "messages": mensagens, "stream": False,
        "options": {"temperature": 0}}).json()["message"]["content"]
    historico.append({"role": "assistant", "content": resposta})
    print(f"🧑 {mensagem}\n🤖 {resposta}")
    # aprender com a troca: o pi-memory extrai fatos e decide guardar/substituir/apagar/ignorar
    decisoes = pm.memories.add(historico[-2:], owner=dono)["decisions"]
    if mostrar_aprendizado:
        for d in decisoes:
            antes = f"  (antes: {d['previous_text']})" if d["previous_text"] else ""
            print(f"   💾 {d['event']}: {d['text']}{antes}")
    print()

## Sessão 1

In [4]:
pm = PiMemory(path=PASTA, llm=OllamaLLM(MODELO, timeout=180))
pm.documents.add(FIXTURES / "contrato_exemplo.pdf", owner="ana")   # um contrato da Ana
historico = []

conversar(pm, "ana", historico, "Oi! Sou a Ana, trabalho como enfermeira e moro em Curitiba.")
conversar(pm, "ana", historico, "Tenho uma gata chamada Mel e prefiro respostas bem curtas.")
conversar(pm, "ana", historico, "No meu contrato, qual o prazo de entrega dos relatórios?")
pm.close()   # "fecha o programa": o histórico da sessão se perde, a memória fica na pasta

🧑 Oi! Sou a Ana, trabalho como enfermeira e moro em Curitiba.
🤖 Olá Ana! É um prazer te conhecer. Como posso te ajudar hoje?


   💾 add: O nome do usuário é Ana
   💾 add: A usuária trabalha como enfermeira
   💾 add: A usuária mora em Curitiba



🧑 Tenho uma gata chamada Mel e prefiro respostas bem curtas.
🤖 Entendido! Vou ser breve. Tudo bem com a Mel? Como posso ajudar?


   💾 add: O usuário tem uma gata chamada Mel
   💾 add: O usuário prefere respostas bem curtas



🧑 No meu contrato, qual o prazo de entrega dos relatórios?
🤖 O prazo é de 45 dias corridos após cada visita [contrato_exemplo.pdf, página 12].


## Sessão 2: outro dia, programa aberto de novo

Histórico vazio. O que o assistente sabe vem só da memória.

In [5]:
pm = PiMemory(path=PASTA, llm=OllamaLLM(MODELO, timeout=180))
historico = []

conversar(pm, "ana", historico, "Bom dia! Você lembra qual é a minha profissão e o nome da minha gata?")
conversar(pm, "ana", historico, "Novidade: me mudei para Recife.")
conversar(pm, "ana", historico, "Em que cidade eu moro agora?")

🧑 Bom dia! Você lembra qual é a minha profissão e o nome da minha gata?
🤖 Bom dia, Ana! Você trabalha como enfermeira e sua gata se chama Mel.


   💾 none: O usuário trabalha como enfermeira
   💾 none: O nome da gata do usuário é Mel



🧑 Novidade: me mudei para Recife.
🤖 Legal, Ana! Agora você mora em Recife.


   💾 update: O usuário se mudou para Recife.  (antes: A usuária mora em Curitiba)



🧑 Em que cidade eu moro agora?
🤖 Você mora em Recife.


## O que ficou guardado

In [6]:
for m in pm.memories.list(owner="ana")["memories"]:
    print(f"- [{m['topic']}] {m['text']}")
pm.close()

- [casa] O usuário se mudou para Recife.
- [perfil] O usuário prefere respostas bem curtas
- [animais] O usuário tem uma gata chamada Mel
- [trabalho] A usuária trabalha como enfermeira
- [perfil] O nome do usuário é Ana


## Variações comuns

- **Guardar de tempos em tempos** em vez de a cada troca: acumule as mensagens e chame
  `memories.add(ultimas_16_mensagens, owner)` a cada N trocas (menos chamadas ao modelo; o
  extrator entende melhor respostas curtas como "sim" com mais contexto).
- **Guardar em segundo plano**: `memories.add` pode rodar numa thread depois de responder, para
  o usuário não esperar a extração.
- **Modelos diferentes**: o modelo que conversa e o que extrai memórias podem ser diferentes
  (`PiMemory(llm=...)` é só o extrator).